# 23 · Performance & Memory

At scale, *how* you write Python matters. This notebook covers measuring before
optimizing, the generators-vs-lists memory tradeoff, chunking big files,
vectorization, and picking efficient data structures.

In [ ]:
# ▶ Run this first. Locates the sample data no matter where the kernel starts.
from pathlib import Path

def find_data() -> Path:
    here = Path.cwd()
    for base in (here, *here.parents):
        if (base / 'data' / 'raw').exists():
            return base / 'data'
    raise FileNotFoundError('Run: uv run python data/build_data.py')

DATA = find_data()
RAW = DATA / 'raw'
print('Data directory:', DATA)
print('Raw files:', sorted(p.name for p in RAW.glob('*')))

## Measure first

Never guess at performance — **profile**. `time.perf_counter` times a block;
`timeit` averages many runs for micro-benchmarks. Optimize the actual
bottleneck, not what you assume it is.

In [ ]:
import timeit

setup = 'data = list(range(1000))'
loop = timeit.timeit('[x*2 for x in data]', setup=setup, number=10000)
vec = timeit.timeit('list(map(lambda x: x*2, data))', setup=setup, number=10000)
print(f'comprehension: {loop:.3f}s')
print(f'map:           {vec:.3f}s')

## Generators vs lists: memory

A list holds every element at once; a generator holds one. For large
intermediate data, generators slash memory. `sys.getsizeof` shows the
difference in the container itself.

In [ ]:
import sys

big_list = [x for x in range(1_000_000)]      # materialized
big_gen = (x for x in range(1_000_000))        # lazy
print('list bytes:', sys.getsizeof(big_list))
print('generator bytes:', sys.getsizeof(big_gen))
print('sum via generator (no big list):', sum(x for x in range(1_000_000)))

## Chunking / streaming large files

When a file is too big for memory, process it in a **streaming** pass — read
one row at a time and aggregate incrementally, never holding the whole file in
memory. Here we stream the CSV with the standard library.

In [ ]:
import csv

total = 0.0
rows = 0
with open(RAW / 'orders.csv', encoding='utf-8', newline='') as f:
    for row in csv.DictReader(f):        # one row in memory at a time
        rows += 1
        if row['status'] == 'completed':
            total += float(row['amount'])
print(f'streamed {rows} rows; revenue={total:,.2f}')

## Choose the right data structure

Membership tests are O(n) in a list but O(1) in a set/dict. For repeated lookups
against a large collection, this is the difference between fast and unusable.

In [ ]:
import time

big = list(range(200_000))
as_set = set(big)
targets = [199_999, -1, 100_000]

t0 = time.perf_counter()
for _ in range(1000):
    _ = [t in big for t in targets]        # list: scans each time
t1 = time.perf_counter()
for _ in range(1000):
    _ = [t in as_set for t in targets]     # set: hash lookup
t2 = time.perf_counter()
print(f'list membership: {(t1 - t0)*1000:.1f} ms')
print(f'set  membership: {(t2 - t1)*1000:.1f} ms')

### Recap

Measure with `timeit`/`perf_counter` before optimizing; generators save memory
on large intermediates; stream big files a row at a time; use sets/dicts for
O(1) membership. That completes the **Python bootcamp** — you can write fast,
correct, production-grade Python. For dataframe wrangling (NumPy + pandas) and a
full ETL capstone, continue to the companion **pandas-numpy-bootcamp**.